# Lecture: Autoencoder for Unsupervised Representation Learning

In the previous lectures, we modeled images either through explicit density estimation (Gaussian) or autoregressive factorization (PixelCNN). Both approaches aim to capture the full data distribution. However, for many practical tasks — such as compression, denoising, or visualization — we are primarily interested in learning a compact, meaningful representation of the data.

An **autoencoder** is a neural network trained to compress data into a low-dimensional **latent space** and then reconstruct it from that representation. It consists of two parts:
- **Encoder**: Maps the input $\mathbf{x} \in \mathbb{R}^N$ to a latent vector $\mathbf{z} \in \mathbb{R}^M$ with $M \ll N$.
- **Decoder**: Maps the latent vector $\mathbf{z}$ back to a reconstruction $\hat{\mathbf{x}} \in \mathbb{R}^N$.

Training minimizes the **reconstruction loss** between the input and its reconstruction — typically mean squared error (MSE). The bottleneck forces the model to learn a compressed representation that captures the most important structure in the data.

When the latent dimension is set to 2, we can directly visualize the latent space and observe how the model organizes different digit classes — without any label supervision during training.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/06-Generative_Image_Models/Autoencoder.py ./

### Data Preparation

We use the full MNIST training set (60,000 images). Each 28×28 grayscale image is normalized to the range [0, 1]. Since the autoencoder reconstructs pixel intensities as continuous values, no quantization is needed — unlike PixelCNN.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -------------------------------------------------
# Device configuration
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------------------------------
# Dataset preparation
# -------------------------------------------------

# Normalize pixel values to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

### Model Architecture

We use a convolutional autoencoder. Convolutional layers exploit the spatial structure of images and are more parameter-efficient than fully connected architectures.

- **Encoder**: Three convolutional blocks (Conv → BatchNorm → ReLU) with stride 2 for downsampling, followed by a linear projection to the latent vector.
- **Decoder**: A linear projection back to the spatial representation, followed by transposed convolutions (ConvTranspose2d) for upsampling, and a final sigmoid activation to constrain outputs to [0, 1].

The latent dimension is set to 2 by default, enabling direct 2D visualization of the learned representation.

In [ ]:
from Autoencoder import Autoencoder

# Verify output shapes
_ae = Autoencoder(latent_dim=2)
_x  = torch.zeros(4, 1, 28, 28)
print("Reconstruction shape:", _ae(_x).shape)      # expected: (4, 1, 28, 28)
print("Latent shape:        ", _ae.encode(_x).shape)  # expected: (4, 2)

### Training

The autoencoder is trained to minimize the mean squared error (MSE) between input images and their reconstructions. MSE penalizes the per-pixel difference and is a natural choice when pixel intensities are continuous values in [0, 1].

Training for 10 epochs on the full MNIST training set takes approximately 2–3 minutes on a Colab GPU.

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

# -------------------------------------------------
# Model initialization
# -------------------------------------------------
LATENT_DIM = 2

model = Autoencoder(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------------------------------
# Training loop
# -------------------------------------------------
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)

        optimizer.zero_grad()
        x_hat = model(x)
        loss = F.mse_loss(x_hat, x)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:2d}, Loss: {total_loss / len(train_loader):.6f}")

In [ ]:
model.save_model()

If you do not want to train, you can load the pre-trained model (latent_dim=2, 10 epochs, full MNIST training set).

In [ ]:
import torch
from Autoencoder import Autoencoder

device = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 2

model = Autoencoder(latent_dim=LATENT_DIM).to(device)
model.load_model(path="AIBIP/06-Generative_Image_Models/models/autoencoder_mnist.pth", device=device)

### Reconstruction Quality

We evaluate the autoencoder qualitatively by comparing original test images with their reconstructions. Each pair shows what information the model successfully preserves through the 2-dimensional bottleneck.

In [ ]:
import matplotlib.pyplot as plt

model.eval()

# Grab one batch from the test set
x_batch, _ = next(iter(test_loader))
x_batch = x_batch[:10].to(device)

with torch.no_grad():
    x_hat = model(x_batch)

# Plot originals (top row) and reconstructions (bottom row)
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].squeeze().cpu(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",       fontsize=10)
axes[1, 0].set_ylabel("Reconstruction", fontsize=10)

plt.tight_layout()
plt.show()

### Latent Space Visualization

Because the latent dimension is 2, we can plot every test image as a point in the 2D latent space, colored by its true digit class. This reveals how the autoencoder organizes the data — entirely without label supervision during training.

Well-separated clusters indicate that the autoencoder has learned class-discriminative features purely from the reconstruction objective.

In [ ]:
import numpy as np

model.eval()

all_z      = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        z = model.encode(x.to(device))
        all_z.append(z.cpu().numpy())
        all_labels.append(y.numpy())

all_z      = np.concatenate(all_z,      axis=0)  # (10000, 2)
all_labels = np.concatenate(all_labels, axis=0)  # (10000,)

# -------------------------------------------------
# Scatter plot
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(
    all_z[:, 0], all_z[:, 1],
    c=all_labels,
    cmap="tab10",
    s=2,
    alpha=0.6
)

plt.colorbar(scatter, ax=ax, label="Digit class", ticks=range(10))
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("2D Latent Space — MNIST Test Set")
plt.tight_layout()
plt.show()

### Latent Space Interpolation

Because the latent space is continuous, we can interpolate linearly between the latent representations of two test images. The decoded images along the path reveal how the model transitions between different digit shapes, demonstrating that the latent space captures smooth, meaningful structure.

In [ ]:
model.eval()

# Pick two test images from different digit classes
test_images, test_labels = next(iter(test_loader))

# Find one sample of digit 1 and one of digit 7
idx_a = (test_labels == 1).nonzero(as_tuple=True)[0][0]
idx_b = (test_labels == 7).nonzero(as_tuple=True)[0][0]

x_a = test_images[idx_a].unsqueeze(0).to(device)  # (1, 1, 28, 28)
x_b = test_images[idx_b].unsqueeze(0).to(device)

with torch.no_grad():
    z_a = model.encode(x_a)  # (1, 2)
    z_b = model.encode(x_b)  # (1, 2)

# Linear interpolation: 10 steps from z_a to z_b
n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img = model.decoder(z_interp).squeeze().cpu()
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}", fontsize=8)

plt.suptitle(f"Interpolation: digit {test_labels[idx_a].item()} → digit {test_labels[idx_b].item()}", y=1.05)
plt.tight_layout()
plt.show()

---

## Sparse Autoencoder

The standard autoencoder places no constraint on the latent space beyond the bottleneck dimension. A **sparse autoencoder** adds an L1 penalty on the encoder activations, forcing most neurons to be inactive for any given input. This encourages each neuron to specialize in a specific visual feature (e.g. "rounded stroke", "vertical line").

The total loss becomes:

$$\mathcal{L} = \underbrace{\|\mathbf{x} - \hat{\mathbf{x}}\|^2}_{\text{reconstruction}} + \lambda \underbrace{\|\mathbf{z}\|_1}_{\text{sparsity}}$$

The sparsity coefficient $\lambda$ controls the trade-off: larger values produce sparser, more interpretable representations at the cost of slightly worse reconstruction.

**What changes compared to the standard autoencoder?**
- The latent vectors are sparse — most entries are near zero for a given input.
- Different digit classes activate different subsets of neurons, improving separability.
- The latent space is *not* structured as well and generation by sampling remains unreliable.

### Model Architecture

We reuse the same convolutional encoder and decoder as before. The only change is the latent dimension — we increase it to **32** so that sparsity has room to create specialised neurons. With only 2 dimensions there is nothing to be sparse over.

In [1]:
from Autoencoder import Autoencoder

SPARSE_LATENT_DIM = 2

sparse_model = Autoencoder(latent_dim=SPARSE_LATENT_DIM).to(device)

_x = torch.zeros(4, 1, 28, 28)
print("Reconstruction shape:", sparse_model(_x).shape)       # (4, 1, 28, 28)
print("Latent shape:        ", sparse_model.encode(_x).shape) # (4, 32)

NameError: name 'device' is not defined

### Training with L1 Sparsity Penalty

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

SPARSITY_LAMBDA = 1e-3  # L1 penalty weight

sparse_optimizer = optim.Adam(sparse_model.parameters(), lr=1e-3)

epochs = 10

for epoch in range(epochs):
    sparse_model.train()
    total_loss = 0

    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)

        sparse_optimizer.zero_grad()
        z = sparse_model.encode(x)
        x_hat = sparse_model.decoder(z)

        recon_loss = F.mse_loss(x_hat, x)
        sparsity_loss = SPARSITY_LAMBDA * z.abs().mean()
        loss = recon_loss + sparsity_loss

        loss.backward()
        sparse_optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:2d}, Loss: {total_loss / len(train_loader):.6f}")

In [ ]:
sparse_model.save_model(path="models/sparse_autoencoder_mnist.pth")

If you do not want to train, you can load the pre-trained sparse autoencoder (latent_dim=32, λ=1e-3, 10 epochs).

In [ ]:
import torch
from Autoencoder import Autoencoder

device = "cuda" if torch.cuda.is_available() else "cpu"
SPARSE_LATENT_DIM = 2

sparse_model = Autoencoder(latent_dim=SPARSE_LATENT_DIM).to(device)
sparse_model.load_model(path="AIBIP/06-Generative_Image_Models/models/sparse_autoencoder_mnist.pth", device=device)

### Reconstruction Quality: Standard vs. Sparse

We compare the reconstruction quality of both models side by side. The sparse autoencoder uses a 32-dimensional latent space (vs. 2D), so its reconstructions should be noticeably sharper — but the sparsity penalty introduces a slight additional cost compared to a plain 32D autoencoder.

In [ ]:
import matplotlib.pyplot as plt

x_batch, _ = next(iter(test_loader))
x_batch = x_batch[:10].to(device)

model.eval()
sparse_model.eval()

with torch.no_grad():
    x_hat_standard = model(x_batch)
    x_hat_sparse   = sparse_model(x_batch)

fig, axes = plt.subplots(3, 10, figsize=(15, 4))
row_labels = ["Original", "Standard AE (2D)", "Sparse AE (32D)"]

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(),        cmap="gray")
    axes[1, i].imshow(x_hat_standard[i].squeeze().cpu(), cmap="gray")
    axes[2, i].imshow(x_hat_sparse[i].squeeze().cpu(),   cmap="gray")
    for row in range(3):
        axes[row, i].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.show()

### Latent Activation Sparsity

To verify that the L1 penalty actually produces sparse activations, we measure the fraction of near-zero latent units (|z| < 0.01) across the test set. A high sparsity ratio confirms that each input activates only a small subset of the 32 neurons.

In [ ]:
import numpy as np

sparse_model.eval()

all_z_sparse = []
all_labels   = []

with torch.no_grad():
    for x, y in test_loader:
        z = sparse_model.encode(x.to(device))
        all_z_sparse.append(z.cpu().numpy())
        all_labels.append(y.numpy())

all_z_sparse = np.concatenate(all_z_sparse, axis=0)  # (10000, 32)
all_labels   = np.concatenate(all_labels,   axis=0)

sparsity_ratio = (np.abs(all_z_sparse) < 0.01).mean()
print(f"Fraction of near-zero activations: {sparsity_ratio:.1%}")

# Mean absolute activation per neuron — shows which neurons are most active
mean_activation = np.abs(all_z_sparse).mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(SPARSE_LATENT_DIM), mean_activation)
ax.set_xlabel("Latent neuron index")
ax.set_ylabel("Mean |z|")
ax.set_title("Per-neuron mean absolute activation (sparse autoencoder)")
plt.tight_layout()
plt.show()

### Latent Space Visualization (PCA projection)

Because the sparse latent space is 32-dimensional, we cannot plot it directly. We apply PCA to project it to 2D for visualization. Even in this compressed view, the class clusters should be more compact and separated than with the 2D standard autoencoder.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
z_2d = pca.fit_transform(all_z_sparse)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard AE — direct 2D latent space (reuse all_z from earlier section)
scatter0 = axes[0].scatter(all_z[:, 0], all_z[:, 1], c=all_labels, cmap="tab10", s=2, alpha=0.5)
axes[0].set_title("Standard AE — 2D latent space")
axes[0].set_xlabel("z[0]")
axes[0].set_ylabel("z[1]")
plt.colorbar(scatter0, ax=axes[0], label="Digit", ticks=range(10))

# Sparse AE — PCA-projected 32D latent space
scatter1 = axes[1].scatter(z_2d[:, 0], z_2d[:, 1], c=all_labels, cmap="tab10", s=2, alpha=0.5)
axes[1].set_title("Sparse AE — PCA projection of 32D latent space")
axes[1].set_xlabel("PC 1")
axes[1].set_ylabel("PC 2")
plt.colorbar(scatter1, ax=axes[1], label="Digit", ticks=range(10))

plt.tight_layout()
plt.show()